# Data Visualization

In [9]:
# Load data set
import pandas as pd
import altair as alt

new_data = pd.read_csv("../data/new_data.csv")
expend = pd.read_csv("../data/expenditures.csv")

In [2]:
# Define Color Palette
colors = ['#4682B4', '#4F7942', '#FFD700', '#CC5500']

## Plot 1: Agricultural Expenses

In [3]:
# Subset Code by attribute
filtered = expend[expend['Attribute'].isin(['Public agricultural and food R&D (constant 2022 U.S. dollars- millions)', 'Private food industry R&D (constant 2022 U.S. dollars- millions)'])]
filtered = filtered[filtered['Year'] != 2022]

# Split attributes
filtered_public = filtered[filtered['Attribute'].isin(['Public agricultural and food R&D (constant 2022 U.S. dollars- millions)'])]
filtered_private = filtered[filtered['Attribute'].isin(['Private food industry R&D (constant 2022 U.S. dollars- millions)'])]

In [ ]:
area1 = alt.Chart(filtered_public).mark_area(color = '#4682B4', opacity=0.5).encode(
    x=alt.X('Year', axis=alt.Axis(format='.0f')),
    y=alt.Y('Value', title='U.S. Dollars (Millions)')
).properties(title='Total Research & Development Expenditure (Public Sector)')

line1 = alt.Chart(filtered_public).mark_line(color = '#4682B4').encode(
    x=alt.X('Year', axis=alt.Axis(format='.0f')),
    y=alt.Y('Value', title='U.S. Dollars (Millions)')
).properties(title='Total Research & Development Expenditure (Public Sector)')

area2 = alt.Chart(filtered_private).mark_area(color = '#4F7942', opacity=0.5).encode(
    x=alt.X('Year', axis=alt.Axis(format='.0f')),
    y=alt.Y('Value', title='U.S. Dollars (Millions)')
).properties(title='Total Research & Development Expenditure (Private Sector)')

line2 = alt.Chart(filtered_private).mark_line(color = '#4F7942').encode(
    x=alt.X('Year', axis=alt.Axis(format='.0f')),
    y=alt.Y('Value', title='U.S. Dollars (Millions)')
).properties(title='Total Research & Development Expenditure (Private Sector)')

chart1 = area1 + line1
chart2 = area2 + line2

combined = chart1 | chart2
combined
combined.save('../website/plots/plot1.svg')

## Plot 2: GE Corn

In [133]:
# Filter data set
filtered = new_data[new_data['Crop'] == "Corn"]
filtered = filtered[filtered['Attribute'] == "All GE varieties (percent of all corn planted) 3/"]
filtered = filtered[~filtered['State'].isin(['Other States', 'United States'])]

In [134]:
state_to_id = {
    "Illinois": 17,
    "Indiana": 18,
    "Iowa": 19,
    "Kansas": 20,
    "Michigan": 26,
    "Minnesota": 27,
    "Missouri": 29,
    "Nebraska": 31,
    "North Dakota": 38,
    "Ohio": 39,
    "South Dakota": 46,
    "Texas": 48,
    "Wisconsin": 55
}
filtered['state_id'] = filtered['State'].map(state_to_id)
filtered['state_id'] = filtered['state_id'].apply(lambda x: int(x))

In [135]:
filtered = filtered.dropna(subset=['Value'])
filtered = filtered[['state_id', 'Value', 'Year']]

In [136]:
from vega_datasets import data

# Load States
states = alt.topo_feature(data.us_10m.url, 'states')

In [137]:
year_slider = alt.param(
    value=2000,
    bind=alt.binding_range(min=2000, max=2025, step=1, name="Year: ")
)

base = alt.Chart(filtered).mark_geoshape().encode(
    color=alt.Color('Value:Q', scale=alt.Scale(range=['#FFD700', '#4F7942', '#4682B4']), title = 'Percent')
).transform_filter(
    alt.datum.Year == year_slider
).transform_lookup(
    lookup='state_id',
    from_=alt.LookupData(states, 'id', ['type', 'properties', 'geometry'])
).project('albersUsa').add_params(year_slider)

borders = alt.Chart(states).mark_geoshape(
    fill=None,
    stroke='black',
    strokeWidth=1
).project('albersUsa')

chart = (base + borders).properties(
    title=alt.TitleParams(text='Genetically Engineered Corn By State (2000-2025)'),
    width = 600,
    height = 400)
chart.save('../website/plots/plot2.html')

# Plot 3: By Crop

In [232]:
# Filter data set
filtered = new_data[new_data['State'] == "United States"]
filtered = filtered[filtered['Attribute'].isin(['All GE varieties (percent of all corn planted) 3/', 'All GE varieties (percent of all upland cotton planted) 3/', 'All GE varieties (percent of all soybeans planted)'])]

In [233]:
# Set color range
color_range = ['#4F7942', '#FFD700', '#CC5500']

In [234]:
# Create selection criteria
select_crop = alt.selection_point(
    fields=["Crop"],
    bind=alt.binding_select(options=list(filtered["Crop"].unique()),name="Select Crop "),
    value="Corn")

# Create chart
chart = (
    alt.Chart(filtered)
    .mark_bar()
    .encode(
        x=alt.X("Year:N", title = "Year", axis=alt.Axis(labelFontSize = 12, titleFontSize=14)),
        y=alt.Y("Value:Q", scale=alt.Scale(domain = [0,100]), title = "Percent of all Planted (%)", axis=alt.Axis(labelFontSize = 12, titleFontSize=14)),
        color=alt.Color("Value:Q", scale=alt.Scale(domain = [0, 50, 100], range = color_range), 
                        title = "Percent", legend = alt.Legend(labelFontSize=12, titleFontSize=12)),
        tooltip=[alt.Tooltip('Value', title = "Percent of all Planted (%)")]
    )
    .add_params(select_crop)
    .transform_filter(select_crop)
    .properties(title = alt.TitleParams(text = 'All Genetically Engineered Varieties By Year And Crop (2000-2025)', subtitle = 'Data Source: USDA, Economic Research Service (ERS)', fontSize=16), width = 500)
)

chart.save('../website/plots/plot3.html')

# Plot 4: By State

In [220]:
# Filter data set
filtered = new_data[new_data['State'] != "United States"]

# Look just at Herbicide-intolerant
filtered = filtered[filtered['Attribute'].isin(['Herbicide-tolerant (HT) only (percent of all corn planted)', 'Herbicide-tolerant (HT) only (percent of all upland cotton planted)', 'Herbicide-tolerant (HT) only (percent of all soybeans planted)'])]
filtered_year = filtered[filtered['Year'] == 2025]

# Reset Index
filtered = filtered.reset_index(drop=True)
filtered_year = filtered_year.reset_index(drop=True)

# Crop
filtered_cotton = filtered[filtered['Crop'] == 'Cotton']
filtered_corn = filtered[filtered['Crop'] == 'Corn']
filtered_soybeans = filtered[filtered['Crop'] == 'Soybeans']

In [221]:
all_values = ['Alabama', 'Arkansas', 'California', 'Corn', 'Cotton', 'Georgia', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Louisiana', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Nebraska', 'North Carolina', 'North Dakota', 'Ohio', 'Other States', 'South Dakota', 'Soybeans', 'Tennessee', 'Texas', 'Wisconsin']
all_colors = ['#111184', '#023020', '#FF8C00', '#4F7942', '#CC5500', '#8B8000', '#0000CD', '#3CB371', '#F08000', '#FFD700', '#87CEFA', '#C1E1C1', '#FFD580', '#FDFD96', '#00CED1', '#008080', '#FF7F50', '#F8DE7E', '#FFCE1B', '#000080', '#0047AB', '#4682B4', '#808000', '#228B22', '#FFA800']
color_scale = alt.Scale(domain=all_values, range=all_colors)

In [231]:
selection = alt.selection_point(name='bar_select', fields=['State', 'Crop'])

bar_chart = alt.Chart(filtered_year).mark_bar().encode(
    x=alt.X('State:N', title='State', axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    y=alt.Y('Value:Q', title='Percent Of All Planted (%)', axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    color=alt.Color('Crop:N', scale=color_scale, legend=None),
    xOffset='Crop:N',
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.2))
).properties(
    title=alt.TitleParams(
        text='Total Herbicide-tolerant (Ht) Only Genetically Engineered Crops (2025)',
        subtitle='Data Source: USDA, Economic Research Service (ERS)',
        fontSize=16),
        width=660,
        height=300
    
).add_params(selection)

line_chart1 = alt.Chart(filtered_corn).mark_line().encode(
    x=alt.X('Year:N', title='Year', axis=alt.Axis(labelFontSize=9)),
    y=alt.Y('Value:Q', title='Percent Of All Planted (%)', axis=alt.Axis(labelFontSize=12, titleFontSize=12)),
    color=alt.Color('State:N', scale=color_scale, legend=None),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.1))
).properties(
    title=alt.TitleParams(
        text='Herbicide-tolerant Corn (2000-2025)',
        fontSize=11),
        width=200,
        height=200
    )

line_chart2 = alt.Chart(filtered_cotton).mark_line().encode(
    x=alt.X('Year:N', title='Year', axis=alt.Axis(labelFontSize=9)),
    y=alt.Y('Value:Q', title=""),
    color=alt.Color('State:N', scale=color_scale, legend=None),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.1))
).properties(
    title=alt.TitleParams(
        text='Herbicide-tolerant Cotton (2000-2025)',
        fontSize=11),
        width=200,
        height=200
    )

line_chart3 = alt.Chart(filtered_soybeans).mark_line().encode(
    x=alt.X('Year:N', title='Year', axis=alt.Axis(labelFontSize=9)),
    y=alt.Y('Value:Q', title = ""),
    color=alt.Color('State:N', scale=color_scale, legend=None),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.1))
).properties(
    title=alt.TitleParams(
        text='Herbicide-tolerant Soybeans (2000-2025)',
        fontSize=11),
        width=200,
        height=200
    )

linked_chart = bar_chart & (line_chart1 | line_chart2 | line_chart3)
linked_chart.save('../website/plots/plot4.html')